In [ ]:
# ¿Unnamed:11 es duplicado numérico de is_blacklisted_sample?
  df_check = df_meta[df_meta['Patient_ID_IF_mod'].notna()].copy()
  print("Unnamed:11 vs is_blacklisted_sample:")
  print(pd.crosstab(df_check['Unnamed: 11'], df_check['is_blacklisted_sample']))

  print("\nUnnamed:15 vs is_blacklisted:")
  print(pd.crosstab(df_check['Unnamed: 15'], df_check['is_blacklisted']))

  # Cuántos en total blacklisted (todos los flags)
  print("\nis_blacklisted (combinado):")
  print(df_check['is_blacklisted'].value_counts())
  print("\nis_blacklisted_sample:")
  print(df_check['is_blacklisted_sample'].value_counts())

In [ ]:
# ¿Qué son exactamente Unnamed: 11 y Unnamed: 15?
print("=== Unnamed: 11 — primeros valores únicos ===")
print(df_meta['Unnamed: 11'].dropna().head(20).to_string())

print("\n=== Unnamed: 15 — primeros valores únicos ===")
print(df_meta['Unnamed: 15'].dropna().head(20).to_string())

# ¿Cuántos pacientes están blacklisted?
print("\n=== BLACKLIST ===")
print(df_meta[df_meta['Patient_ID_IF_mod'].notna()]['is_blacklisted'].value_counts())
print("\nis_blacklisted_sample:")
print(df_meta[df_meta['Patient_ID_IF_mod'].notna()]['is_blacklisted_sample'].value_counts())
print("\nis_blacklisted_cohort:")
print(df_meta[df_meta['Patient_ID_IF_mod'].notna()]['is_blacklisted_cohort'].value_counts())

In [ ]:
# Encontrar la fila fantasma (la que tiene texto de cabecera como dato)
fila_fantasma = df_meta[df_meta['Patient_ID_IF_mod'].isna()]
print(f"Filas con Patient_ID_IF_mod nulo: {len(fila_fantasma)}")
print(fila_fantasma.T)  # Ver toda la fila transpuesta

# Ver qué hay en las Unnamed
print("\n=== Unnamed: 11 ===")
print(df_meta['Unnamed: 11'].value_counts(dropna=False).head(10))
print("\n=== Unnamed: 15 ===")
print(df_meta['Unnamed: 15'].value_counts(dropna=False).head(10))

# Ver columnas que faltan en el resumen anterior
print("\n=== COLUMNAS RESTANTES (las que no salieron) ===")
print(df_meta.columns.tolist())

In [ ]:
# Ver TODAS las columnas y sus nulos
print("=== TODAS LAS COLUMNAS DE METADATOS ===")
nulos_meta = df_meta.isnull().sum()
pct = (nulos_meta / len(df_meta) * 100).round(1)
resumen = pd.DataFrame({'nulos': nulos_meta, 'pct_nulos': pct, 'tipo': df_meta.dtypes})
print(resumen.to_string())

# Columnas Unnamed
print("\n=== COLUMNAS UNNAMED ===")
unnamed = [c for c in df_meta.columns if 'Unnamed' in str(c)]
print(df_meta[unnamed].head(10))

# Columnas _load que son object — ver qué valores raros tienen
print("\n=== VALORES RAROS EN COLUMNAS _load ===")
load_cols = [c for c in df_meta.columns if '_load' in c]
for col in load_cols:
    no_num = df_meta[col][pd.to_numeric(df_meta[col], errors='coerce').isna() & df_meta[col].notna()]
    if len(no_num) > 0:
        print(f"{col}: {no_num.unique()[:5]}")
    else:
        print(f"{col}: OK (convertible a número)")

In [ ]:
import pandas as pd

df_meta = pd.read_excel("../Metadatos_to_ALV.xlsx")

print(f"Shape: {df_meta.shape}")
print(f"\nTipos de datos:")
print(df_meta.dtypes.value_counts())
print(f"\nNulos por columna:")
nulos = df_meta.isnull().sum()
print(nulos[nulos > 0].sort_values(ascending=False))
print(f"\nColumnas sin ningún nulo: {(nulos == 0).sum()}")
print(f"\nPrimeras columnas:")

print(df_meta.dtypes.to_string())

In [ ]:
print("Pacientes únicos en Tumor_blood:", df_sheet1["Patient_ID_IF_mod"].nunique())
print("Filas totales en Tumor_blood:", len(df_sheet1))

# Ver cuántas filas tiene cada paciente
filas_por_paciente = df_sheet1["Patient_ID_IF_mod"].value_counts()
print("\nDistribución de filas por paciente:")
print(filas_por_paciente.value_counts().sort_index())

In [ ]:
ids_meta = set(df_meta["Patient_ID_IF_mod"])
ids_tumor = set(df_sheet1["Patient_ID_IF_mod"])
ids_drivers = set(df_drivers["Patient_ID_IF_mod"])

print(f"Pacientes en Metadatos:    {len(ids_meta)}")
print(f"Pacientes en Tumor_blood:  {len(ids_tumor)}")
print(f"Pacientes en Drivers:      {len(ids_drivers)}")

comunes = ids_meta & ids_tumor & ids_drivers
print(f"\nPacientes en los TRES archivos: {len(comunes)}")

solo_meta_tumor = ids_meta & ids_tumor
print(f"Pacientes en Metadatos Y Tumor_blood: {len(solo_meta_tumor)}")

In [ ]:
# Pacientes con mutación en ATRX o DAXX
atrx_daxx = df_drivers[df_drivers["gene"].isin(["ATRX", "DAXX"])]

print("=== MUTACIONES ATRX / DAXX ===")
print(atrx_daxx["gene"].value_counts())

print(f"\nPacientes únicos con ATRX o DAXX mutado: {atrx_daxx['Patient_ID_IF_mod'].nunique()}")
print(f"Total pacientes en el dataset: {df_drivers['Patient_ID_IF_mod'].nunique()}")

In [ ]:
# Los 20 pacientes con mayor tasa outward
top20 = df_sheet1[["Patient_ID_IF_mod", "Tumor_TF rate OUTWARDS",
                      "Tumor_TF rate INWARDS", "cancer_type"]].sort_values(
      "Tumor_TF rate OUTWARDS", ascending=False).head(20)

print(top20.to_string())

In [ ]:
outward = df_sheet1["Tumor_TF rate OUTWARDS"]

print("=== ESTADÍSTICAS BÁSICAS ===")
print(outward.describe())

print("\n=== ¿CUÁNTOS PACIENTES SUPERAN ESTOS UMBRALES? ===")
for umbral in [0.1, 0.5, 1, 2, 5, 10, 20]:
      n = (outward > umbral).sum()
      pct = (outward > umbral).mean() * 100
      print(f"  > {umbral:4}: {n:4} pacientes ({pct:.1f}%)")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gráfica 1: distribución completa
df_sheet1["Tumor_TF rate OUTWARDS"].hist(bins=100, ax=axes[0])
axes[0].set_title("Distribución fusiones outward (completa)")
axes[0].set_xlabel("Tasa de fusiones outward")
axes[0].set_ylabel("Número de pacientes")

# Gráfica 2: zoom en los valores bajos (sin outliers extremos)
df_sheet1[df_sheet1["Tumor_TF rate OUTWARDS"] < 10]["Tumor_TF rate OUTWARDS"].hist(bins=100, ax=axes[1])
axes[1].set_title("Distribución fusiones outward (zoom < 10)")
axes[1].set_xlabel("Tasa de fusiones outward")
axes[1].set_ylabel("Número de pacientes")

plt.tight_layout()
plt.show()

In [ ]:
 # Buscar "alt" en todos los nombres de columna de todos los archivos
archivos = {
      "Metadatos": df_meta,
      "Tumor_blood Sheet1": df_sheet1,
      "Drivers Sheet1": df_drivers
  }

print("Buscando columnas con 'alt', 'alv', 'tmm', 'telom', 'mech':\n")
for nombre, df in archivos.items():
      encontradas = [c for c in df.columns if any(
          kw in c.lower() for kw in ["alt", "alv", "tmm", "telom", "mech", "label", "class", "status", "positive"]
      )]
      if encontradas:
          print(f"{nombre}: {encontradas}")
      else:
          print(f"{nombre}: ninguna")

In [ ]:
 # Confirmar ATRX y DAXX
print("¿Está ATRX?", "ATRX" in df_drivers["gene"].values)
print("¿Está DAXX?", "DAXX" in df_drivers["gene"].values)

# Ver valores únicos de columnas clave
print("\nVALORES EN category:")
print(df_drivers["category"].unique())

print("\nVALORES EN driver:")
print(df_drivers["driver"].unique())

print("\nVALORES EN Mixed_def:")
print(df_drivers["Mixed_def"].unique())

In [ ]:
df_drivers = pd.read_excel("../Drivers_TF_TVR_TL_to_ALV.xlsx", sheet_name="Sheet1")

# Ver todas las columnas
print("COLUMNAS:")
print(df_drivers.columns.tolist())

print("\nVALORES ÚNICOS EN gene:")
print(df_drivers["gene"].unique())

In [ ]:
for sheet in xl_drivers.sheet_names:
      df = pd.read_excel("../Drivers_TF_TVR_TL_to_ALV.xlsx", sheet_name=sheet, nrows=2)
      print(f"--- {sheet} --- shape:")
      df_full = pd.read_excel("../Drivers_TF_TVR_TL_to_ALV.xlsx", sheet_name=sheet)
      print(f"Filas: {df_full.shape[0]}, Columnas: {df_full.shape[1]}")
      print(df_full.columns.tolist()[:20])
      print()

In [ ]:
xl_drivers = pd.ExcelFile("../Drivers_TF_TVR_TL_to_ALV.xlsx")
print(xl_drivers.sheet_names)

In [ ]:
df_sheet1 = pd.read_excel("../Tumor_blood_to_ALV_2.xlsx", sheet_name="Sheet1")
print(df_sheet1[["Patient_ID_IF_mod", "Tumor_TF rate OUTWARDS", "Blood_TF rate OUTWARDS"]].head(10))

In [ ]:
df_sheet4 = pd.read_excel("../Tumor_blood_to_ALV_2.xlsx", sheet_name="Sheet4")
print(df_sheet4)

In [ ]:
for sheet in xl.sheet_names:
    df = pd.read_excel("../Tumor_blood_to_ALV_2.xlsx", sheet_name=sheet)
    print(f"--- {sheet} --- shape: {df.shape}")
    print(df.columns.tolist())
    print()

In [ ]:
 # Ver las hojas que tiene el archivo
xl = pd.ExcelFile("../Tumor_blood_to_ALV_2.xlsx")
print(xl.sheet_names)

In [ ]:
# Buscar columnas que contengan "alt" o "ALT" o "tmm" o "telom"
keywords = ["alt", "tmm", "telom", "status", "mechanism"]

for col in df_meta.columns:
    for kw in keywords:
        if kw.lower() in col.lower():
            print(col)
            break

In [ ]:
print(df_meta.columns.tolist())

In [ ]:
import subprocess
subprocess.run(["pip", "install", "openpyxl"])

In [ ]:
import pandas as pd

df_meta = pd.read_excel("../Metadatos_to_ALV.xlsx")

print(df_meta.shape)